# Lesson 20: Building with Small Language Models (SLMs)

In this notebook, you will:
1. Run an open-weight Instruct SLM (`google/gemma-2-2b-it` or `Qwen/Qwen2.5-0.5B-Instruct`) with Hugging Face `transformers`.
2. Extract distinct keyframes from synthetic video frames using **OpenCV** (`cv2.calcHist` + `cv2.compareHist < 0.9`).
3. Inspect Sparse MoE top-2 expert routing.

In [ ]:
!pip install -q -U transformers accelerate opencv-python-headless numpy torch

In [ ]:
import cv2, numpy as np

# Synthesize 3 video frames (two similar blue frames, one sudden orange scene change)
frames = [
    np.full((64, 64, 3), [200, 50, 20], dtype=np.uint8),
    np.full((64, 64, 3), [195, 55, 25], dtype=np.uint8),
    np.full((64, 64, 3), [20, 140, 240], dtype=np.uint8),
]

prev_hist = None
keyframes = []
for i, frame in enumerate(frames):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    if prev_hist is None or cv2.compareHist(prev_hist, hist, cv2.HISTCMP_CORREL) < 0.9:
        keyframes.append(i)
        prev_hist = hist

print("Extracted Keyframe Indices:", keyframes)

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
out = gen("Explain why Small Language Models are great for edge devices in one sentence.", max_new_tokens=50)
print(out[0]["generated_text"])